In [1]:
from pathlib import Path

import joblib
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

train_df = pd.read_parquet("../artifacts/03_train.parquet")
val_df = pd.read_parquet("../artifacts/03_validation.parquet")
test_df = pd.read_parquet("../artifacts/03_test.parquet")

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

Train: (67533, 15)
Validation: (14471, 15)
Test: (14472, 15)


In [2]:
def create_features(df):
    df = df.copy()

    purchase_time = pd.to_datetime(df["order_purchase_timestamp"], errors="coerce")

    df["purchase_year"] = purchase_time.dt.year
    df["purchase_month"] = purchase_time.dt.month
    df["purchase_day"] = purchase_time.dt.day
    df["purchase_weekday"] = purchase_time.dt.weekday
    df["purchase_hour"] = purchase_time.dt.hour

    return df


train_df = create_features(train_df)
val_df = create_features(val_df)
test_df = create_features(test_df)

print(train_df.shape)
print(train_df.columns.tolist())

(67533, 20)
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'customer_zip_code_prefix_x', 'customer_city_x', 'customer_state_x', 'customer_zip_code_prefix_y', 'customer_city_y', 'customer_state_y', 'late_delivery', 'purchase_year', 'purchase_month', 'purchase_day', 'purchase_weekday', 'purchase_hour']


In [3]:
target = "late_delivery"

# Columns that should NOT be model inputs
drop_cols = [
    target,
    "order_id",
    "customer_id",
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    "order_status",
    "review_score",
    "review_count",
]

X_train = train_df.drop(columns=drop_cols, errors="ignore")
y_train = train_df[target]

X_val = val_df.drop(columns=drop_cols, errors="ignore")
y_val = val_df[target]

X_test = test_df.drop(columns=drop_cols, errors="ignore")
y_test = test_df[target]

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

X_train: (67533, 11)
X_val: (14471, 11)
X_test: (14472, 11)


In [6]:
numeric_features = X_train.select_dtypes(include=["number"]).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object", "str", "category", "bool"]
).columns.tolist()

print("Numerical features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

Numerical features:
['customer_zip_code_prefix_x', 'customer_zip_code_prefix_y', 'purchase_year', 'purchase_month', 'purchase_day', 'purchase_weekday', 'purchase_hour']

Categorical features:
['customer_city_x', 'customer_state_x', 'customer_city_y', 'customer_state_y']


In [7]:
numeric_pipeline = Pipeline(
    [("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]
)

categorical_pipeline = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]
)

preprocessor = ColumnTransformer(
    [
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features),
    ]
)

In [8]:
X_train_processed = preprocessor.fit_transform(X_train)

X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

print("Processed train:", X_train_processed.shape)
print("Processed validation:", X_val_processed.shape)
print("Processed test:", X_test_processed.shape)

Processed train: (67533, 7361)
Processed validation: (14471, 7361)
Processed test: (14472, 7361)


In [9]:
feature_names = preprocessor.get_feature_names_out()

print("Final feature count:", len(feature_names))
print("\nFirst 30 features:")
print(feature_names[:30])

Final feature count: 7361

First 30 features:
['numeric__customer_zip_code_prefix_x'
 'numeric__customer_zip_code_prefix_y' 'numeric__purchase_year'
 'numeric__purchase_month' 'numeric__purchase_day'
 'numeric__purchase_weekday' 'numeric__purchase_hour'
 'categorical__customer_city_x_abadia dos dourados'
 'categorical__customer_city_x_abadiania'
 'categorical__customer_city_x_abaete'
 'categorical__customer_city_x_abaetetuba'
 'categorical__customer_city_x_abaiara'
 'categorical__customer_city_x_abaira'
 'categorical__customer_city_x_abare'
 'categorical__customer_city_x_abatia'
 'categorical__customer_city_x_abelardo luz'
 'categorical__customer_city_x_abrantes'
 'categorical__customer_city_x_abre campo'
 'categorical__customer_city_x_abreu e lima'
 'categorical__customer_city_x_acaiaca'
 'categorical__customer_city_x_acailandia'
 'categorical__customer_city_x_acarau'
 'categorical__customer_city_x_acari'
 'categorical__customer_city_x_acegua'
 'categorical__customer_city_x_acopiara'


In [12]:
X_train_processed = pd.DataFrame(
    X_train_processed, columns=feature_names, index=train_df.index
)

X_val_processed = pd.DataFrame(
    X_val_processed, columns=feature_names, index=val_df.index
)

X_test_processed = pd.DataFrame(
    X_test_processed, columns=feature_names, index=test_df.index
)

print(X_train_processed.shape)
print(X_val_processed.shape)
print(X_test_processed.shape)

(67533, 7361)
(14471, 7361)
(14472, 7361)


In [13]:
train_features = X_train_processed.copy()
train_features[target] = y_train.values

val_features = X_val_processed.copy()
val_features[target] = y_val.values

test_features = X_test_processed.copy()
test_features[target] = y_test.values

print("Train features:", train_features.shape)
print("Validation features:", val_features.shape)
print("Test features:", test_features.shape)

Train features: (67533, 7362)
Validation features: (14471, 7362)
Test features: (14472, 7362)


In [14]:
artifacts_dir = Path("../artifacts")
artifacts_dir.mkdir(exist_ok=True)

train_features.to_parquet(artifacts_dir / "05_train_features.parquet", index=False)

val_features.to_parquet(artifacts_dir / "05_validation_features.parquet", index=False)

test_features.to_parquet(artifacts_dir / "05_test_features.parquet", index=False)

print("Feature datasets saved.")

Feature datasets saved.


In [15]:
joblib.dump(preprocessor, artifacts_dir / "05_preprocessor.joblib")

print("Saved: ../artifacts/05_preprocessor.joblib")

Saved: ../artifacts/05_preprocessor.joblib


In [16]:
feature_list = pd.DataFrame({"feature": feature_names})

feature_list.to_csv(artifacts_dir / "05_feature_list.csv", index=False)

print("Saved: ../artifacts/05_feature_list.csv")

Saved: ../artifacts/05_feature_list.csv
